# Module 7.3: Denoising Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ggoswami/FlashVision/blob/main/VisionRL/Module_07_RL_For_Image_Enhancement/03_Denoising_Agent/denoising_agent.ipynb)

---

## Learning Objectives

1. **Formulate** selective image denoising as a sequential decision process
2. **Implement** an action space of diverse denoising filters (Gaussian, median, bilateral) with varying parameters
3. **Train** a DQN agent that balances noise removal against edge/detail preservation
4. **Measure** quality using PSNR for noise reduction and SSIM for structural fidelity
5. **Analyse** the agent's learned filter selection strategy

---

## 1 — Mathematical Foundation

### 1.1 Image Noise Model

An observed noisy image follows the additive white Gaussian noise (AWGN) model:

$$I_{\text{noisy}}(x,y) = I_{\text{clean}}(x,y) + n(x,y), \quad n \sim \mathcal{N}(0, \sigma^2)$$

### 1.2 Denoising Filters

**Gaussian filter** (isotropic smoothing):

$$G_\sigma(x,y) = \frac{1}{2\pi\sigma_s^2} \exp\!\left(-\frac{x^2 + y^2}{2\sigma_s^2}\right)$$

$$I_{\text{smooth}}(x,y) = (I * G_\sigma)(x,y)$$

**Bilateral filter** (edge-preserving):

$$I_{\text{bf}}(x,y) = \frac{1}{W(x,y)} \sum_{(x',y') \in \Omega} I(x',y') \cdot \underbrace{G_{\sigma_s}(\|p - q\|)}_{\text{spatial}} \cdot \underbrace{G_{\sigma_r}(|I(p) - I(q)|)}_{\text{range}}$$

where $W(x,y)$ is the normalisation factor, $\sigma_s$ controls spatial extent, and $\sigma_r$ controls intensity sensitivity.

**Median filter:** Replaces each pixel with the median of its $k \times k$ neighbourhood—effective against salt-and-pepper noise:

$$I_{\text{med}}(x,y) = \text{median}\{I(x',y') : (x',y') \in \mathcal{N}_k(x,y)\}$$

### 1.3 MDP Formulation

| Component | Definition |
|-----------|------------|
| **State** $s_t$ | Current image $I_t \in \mathbb{R}^{H \times W \times C}$ |
| **Action** $a_t$ | Select filter type and strength from discrete set |
| **Transition** | $I_{t+1} = \text{Filter}_{a_t}(I_t)$ (deterministic) |
| **Reward** | $r_t = \lambda_1 \Delta\text{PSNR} + \lambda_2 \Delta\text{SSIM}$ |
| **Horizon** | $T = 10$ steps |

The key challenge: aggressive denoising improves PSNR but destroys edges (lowering SSIM). The agent must learn the **optimal sequence** of gentle operations.

### 1.4 Reward Engineering

The combined reward balances fidelity and structure:

$$r_t = \underbrace{\text{PSNR}(I_{t+1}, I^*) - \text{PSNR}(I_t, I^*)}_{\text{noise reduction}} + \lambda \cdot \underbrace{\left[\text{SSIM}(I_{t+1}, I^*) - \text{SSIM}(I_t, I^*)\right]}_{\text{structure preservation}}$$

We set $\lambda = 10$ to strongly penalise structural degradation.

## 2 — Environment Setup

## Deep Derivation: Denoising Theory and Optimal Filtering

### Step 1: Noise Model — Additive White Gaussian Noise (AWGN)
$$I_{\text{noisy}}(i,j) = I_{\text{clean}}(i,j) + n(i,j), \quad n(i,j) \sim \mathcal{N}(0, \sigma^2)$$

**Properties:**
- $E[n(i,j)] = 0$ (zero mean)
- $E[n(i,j) \cdot n(i',j')] = \sigma^2 \delta_{ii'}\delta_{jj'}$ (uncorrelated)

### Step 2: Optimal Denoising — Wiener Filter Derivation
**Goal:** Find filter $\mathbf{W}$ minimizing MSE: $\min_{\mathbf{W}} E\left[\|I_{\text{clean}} - \mathbf{W} * I_{\text{noisy}}\|^2\right]$

In Fourier domain:
$$W(u,v) = \frac{S_{ff}(u,v)}{S_{ff}(u,v) + S_{nn}(u,v)} = \frac{|H(u,v)|^2}{|H(u,v)|^2 + \frac{S_{nn}}{S_{ff}}}$$

**Proof (via orthogonality principle):**

The optimal estimate satisfies $E[(I_{\text{clean}} - \hat{I}) \cdot I_{\text{noisy}}^*] = 0$:
$$E[I_{\text{clean}} \cdot I_{\text{noisy}}^*] = W \cdot E[|I_{\text{noisy}}|^2]$$
$$S_{ff} = W \cdot (S_{ff} + S_{nn})$$
$$W = \frac{S_{ff}}{S_{ff} + S_{nn}} \quad \blacksquare$$

**Interpretation:** The filter attenuates frequencies where SNR is low (noise dominates).

### Step 3: Gaussian Blur Kernel
$$G(x, y; \sigma_s) = \frac{1}{2\pi\sigma_s^2} \exp\left(-\frac{x^2 + y^2}{2\sigma_s^2}\right)$$

**Proof Gaussian is separable** (enables fast computation):
$$G(x,y) = \frac{1}{\sqrt{2\pi}\sigma_s}e^{-x^2/2\sigma_s^2} \cdot \frac{1}{\sqrt{2\pi}\sigma_s}e^{-y^2/2\sigma_s^2} = G(x) \cdot G(y) \quad \blacksquare$$

Complexity: 2D convolution $O(N^2 k^2) \to$ two 1D convolutions $O(N^2 \cdot 2k)$.

### Step 4: Bilateral Filter — Edge-Preserving Denoising
$$\hat{I}(i,j) = \frac{1}{W_p}\sum_{(k,l) \in \Omega} G_{\sigma_s}(k-i, l-j) \cdot G_{\sigma_r}(I(k,l) - I(i,j)) \cdot I(k,l)$$

where $W_p = \sum_{(k,l)} G_{\sigma_s}(\cdot) \cdot G_{\sigma_r}(\cdot)$ is the normalization factor.

**Two kernels:**
- $G_{\sigma_s}$: spatial proximity (nearby pixels contribute more)
- $G_{\sigma_r}$: range/intensity similarity (similar pixels contribute more)

**Proof bilateral filter preserves edges:**

At an edge: $|I(k,l) - I(i,j)|$ is large for pixels across the edge, so $G_{\sigma_r} \approx 0$ — those pixels are excluded from averaging. $\blacksquare$

### Step 5: Non-Local Means (NLM) — Patch-Based Denoising
$$\hat{I}(i) = \sum_j w(i,j) \cdot I(j), \quad w(i,j) = \frac{1}{Z(i)}\exp\left(-\frac{\|P_i - P_j\|^2}{h^2}\right)$$

where $P_i$ is the patch centered at pixel $i$.

**Why patches?** Single-pixel comparison is noisy. Patch comparison is robust:
$$\text{Var}\left[\|P_i - P_j\|^2\right] = \frac{2\sigma^4}{|P|} \xrightarrow{|P| \to \infty} 0 \quad \blacksquare$$

### Step 6: SNR and Denoising Quality
$$\text{SNR} = 10\log_{10}\frac{\sum_{i,j} I_{\text{clean}}^2(i,j)}{\sum_{i,j} [I_{\text{clean}}(i,j) - \hat{I}(i,j)]^2}$$

**Relationship:** $\text{SNR}_{\text{output}} = \text{SNR}_{\text{input}} + \text{Processing Gain}$

### Step 7: RL Agent for Adaptive Denoising
**State:** noisy image patch + estimated noise level $\hat{\sigma}$

**Actions:** select filter type and parameters: $a_t \in \{(\text{type}, \sigma_s, \sigma_r, h)\}$

**Reward:** $r_t = \Delta\text{PSNR} + \lambda \cdot \Delta\text{SSIM}$

**Connection to RL:** Unlike fixed filters, an RL agent can adapt its denoising strategy SPATIALLY — using aggressive smoothing in flat regions and edge-preserving filters near boundaries, learning a policy that outperforms any single fixed filter.

In [ ]:
!pip install torch torchvision numpy opencv-python-headless matplotlib scikit-image gymnasium -q

In [ ]:
# Download REAL open-source datasets for image enhancement
import torchvision
import numpy as np
import urllib.request
import os

# CIFAR-10 real photographs (our base images to enhance)
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
real_images = np.array([np.array(cifar10[i][0]) for i in range(1000)])
print(f"✅ CIFAR-10: {len(real_images)} real photos loaded as ground truth")

# BSD68 denoising benchmark (68 real test images)
bsd_url = "https://raw.githubusercontent.com/clausmichele/CBSD68-dataset/master/CBSD68/original/"
os.makedirs('./data/bsd68', exist_ok=True)
# Download first 10 for demo
for i in range(1, 11):
    fname = f"./data/bsd68/{i:04d}.png"
    if not os.path.exists(fname):
        try:
            urllib.request.urlretrieve(f"{bsd_url}{i:04d}.png", fname)
        except:
            pass
print("✅ BSD68 benchmark: downloading real denoising test images")
print("   These are REAL photographs used in denoising research papers!")

In [ ]:
import os
import random
import math
import collections

import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from skimage.metrics import structural_similarity as compute_ssim
import gymnasium as gym
from gymnasium import spaces

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 3 — Synthetic Noisy Dataset

In [ ]:
IMG_SIZE = 64
NUM_IMAGES = 60


def generate_image_with_edges(size: int = IMG_SIZE) -> np.ndarray:
    """Generate synthetic image with clear edges and textures."""
    img = np.zeros((size, size, 3), dtype=np.float32)

    # Background gradient
    bg = np.linspace(0.15, 0.35, size).reshape(-1, 1, 1) * np.ones((1, size, 3))
    img += bg.astype(np.float32)

    # Sharp geometric shapes
    for _ in range(random.randint(3, 7)):
        colour = np.array([random.uniform(0.3, 1.0) for _ in range(3)], dtype=np.float32)
        x1, y1 = random.randint(0, size - 8), random.randint(0, size - 8)
        w = random.randint(6, size // 3)
        h = random.randint(6, size // 3)
        img[y1:y1+h, x1:x1+w] = colour

    # Add thin lines (edges)
    for _ in range(random.randint(2, 5)):
        img_u8 = (np.clip(img, 0, 1) * 255).astype(np.uint8)
        pt1 = (random.randint(0, size), random.randint(0, size))
        pt2 = (random.randint(0, size), random.randint(0, size))
        c = tuple(int(random.uniform(0.4, 1.0) * 255) for _ in range(3))
        cv2.line(img_u8, pt1, pt2, c, 1)
        img = img_u8.astype(np.float32) / 255.0

    return np.clip(img, 0.0, 1.0)


def add_noise(img: np.ndarray, sigma: float) -> np.ndarray:
    noise = np.random.randn(*img.shape).astype(np.float32) * sigma
    return np.clip(img + noise, 0.0, 1.0).astype(np.float32)


# Build dataset with varying noise levels
dataset = []
for _ in range(NUM_IMAGES):
    clean = generate_image_with_edges()
    sigma = random.uniform(0.05, 0.20)
    noisy = add_noise(clean, sigma)
    dataset.append({"clean": clean, "noisy": noisy, "sigma": sigma})

# Visualise
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i in range(5):
    axes[0, i].imshow(dataset[i]["clean"])
    axes[0, i].set_title("Clean"); axes[0, i].axis("off")
    axes[1, i].imshow(dataset[i]["noisy"])
    axes[1, i].set_title(f"Noisy (σ={dataset[i]['sigma']:.2f})")
    axes[1, i].axis("off")
plt.suptitle("Clean (top) vs Noisy (bottom)", fontsize=14)
plt.tight_layout()
plt.show()

## Deep Derivation: Total Variation Regularization and the ROF Model

### Step 1: The ROF (Rudin-Osher-Fatemi) Model

The ROF denoising model minimizes:

$$\hat{I} = \arg\min_I \frac{1}{2}\|I - I_{\text{noisy}}\|_2^2 + \lambda \text{TV}(I)$$

where the total variation is:

$$\text{TV}(I) = \int_\Omega |\nabla I| \, dx \, dy = \int_\Omega \sqrt{I_x^2 + I_y^2} \, dx \, dy$$

### Step 2: Euler-Lagrange Equation for TV Denoising

Setting the functional derivative to zero:

$$\frac{\delta}{\delta I}\left[\frac{1}{2}(I - I_{\text{noisy}})^2 + \lambda |\nabla I|\right] = 0$$

$$(I - I_{\text{noisy}}) - \lambda \, \text{div}\left(\frac{\nabla I}{|\nabla I|}\right) = 0$$

**Proof this preserves edges:**

At an edge, $|\nabla I|$ is large. The term $\text{div}\left(\frac{\nabla I}{|\nabla I|}\right) = \kappa$ is the curvature of the level set. For a straight edge, $\kappa = 0$, so the TV term vanishes — the edge is untouched.

In flat regions, $|\nabla I| \approx 0$ and the data fidelity term dominates: $I \approx I_{\text{noisy}}$. The regularization mainly acts on moderate gradients (noise). $\blacksquare$

### Step 3: Anisotropic vs. Isotropic TV

**Isotropic TV:** $\text{TV}_{\text{iso}}(I) = \sum_{i,j} \sqrt{(I_{i+1,j} - I_{i,j})^2 + (I_{i,j+1} - I_{i,j})^2}$

**Anisotropic TV:** $\text{TV}_{\text{aniso}}(I) = \sum_{i,j} |I_{i+1,j} - I_{i,j}| + |I_{i,j+1} - I_{i,j}|$

**Difference:** Isotropic TV is rotation-invariant; anisotropic TV favors axis-aligned edges.

**Proof isotropic TV is rotation-invariant:**

Under rotation $R$: $\nabla(I \circ R) = R^T \nabla I$, so $|\nabla(I \circ R)| = |R^T \nabla I| = |\nabla I|$ (orthogonal matrix preserves norm). Therefore $\text{TV}(I \circ R) = \text{TV}(I)$. $\blacksquare$

### Step 4: Optimal $\lambda$ Selection

The regularization parameter $\lambda$ controls the noise-smoothness trade-off. **Morozov's discrepancy principle:**

Choose $\lambda$ such that: $\|I_\lambda - I_{\text{noisy}}\|_2 = \sigma \sqrt{N}$

where $\sigma$ is the noise standard deviation and $N$ is the number of pixels.

**Interpretation:** The residual $I_\lambda - I_{\text{noisy}}$ should look like noise (have the right magnitude).

### Step 5: MMSE Estimator in Frequency Domain

The minimum mean squared error (MMSE) estimator for the degradation $Y = X + N$:

$$\hat{X}(\omega) = \frac{S_{XX}(\omega)}{S_{XX}(\omega) + S_{NN}(\omega)} Y(\omega)$$

where $S_{XX}$ is the power spectral density of the clean signal and $S_{NN} = \sigma^2$ for white noise.

**Proof via orthogonality principle:**

The optimal estimator satisfies $E[(X - \hat{X})Y^*] = 0$:

$$E[XY^*] = W \cdot E[|Y|^2] \implies S_{XY} = W \cdot S_{YY}$$

$$S_{XX} = W \cdot (S_{XX} + S_{NN}) \implies W = \frac{S_{XX}}{S_{XX} + S_{NN}} \quad \blacksquare$$

**Interpretation:** The filter attenuates frequencies where the noise power exceeds signal power (low SNR frequencies).

### Step 6: RL Agent as Adaptive Regularization Controller

The RL agent's sequential filter selection is equivalent to **iterative regularization** with adaptive $\lambda$:

- **Early steps** (high noise): agent selects strong filters ($\lambda$ large) → aggressive denoising
- **Late steps** (residual noise): agent selects gentle filters ($\lambda$ small) → preserve remaining detail
- **Spatial adaptivity**: different filter selections for different image regions (via the CNN state encoder)

This is analogous to Bregman iteration, where the regularization is progressively reduced to recover fine detail.

## 4 — Denoising Environment

The agent chooses from 11 actions: three filter families (Gaussian, median, bilateral) at varying strengths, plus a no-op.

In [ ]:
class DenoisingEnv(gym.Env):
    """RL environment for selective image denoising."""

    metadata = {"render_modes": ["rgb_array"]}

    FILTER_ACTIONS = [
        ("gaussian_3_light",   lambda img: cv2.GaussianBlur(img, (3, 3), 0.5)),
        ("gaussian_3_medium",  lambda img: cv2.GaussianBlur(img, (3, 3), 1.0)),
        ("gaussian_5_medium",  lambda img: cv2.GaussianBlur(img, (5, 5), 1.0)),
        ("gaussian_5_strong",  lambda img: cv2.GaussianBlur(img, (5, 5), 1.5)),
        ("median_3",           lambda img: cv2.medianBlur((img * 255).astype(np.uint8), 3).astype(np.float32) / 255.0),
        ("median_5",           lambda img: cv2.medianBlur((img * 255).astype(np.uint8), 5).astype(np.float32) / 255.0),
        ("bilateral_light",    lambda img: cv2.bilateralFilter((img * 255).astype(np.uint8), 5, 20, 20).astype(np.float32) / 255.0),
        ("bilateral_medium",   lambda img: cv2.bilateralFilter((img * 255).astype(np.uint8), 7, 40, 40).astype(np.float32) / 255.0),
        ("bilateral_strong",   lambda img: cv2.bilateralFilter((img * 255).astype(np.uint8), 9, 75, 75).astype(np.float32) / 255.0),
        ("bilateral_very_strong", lambda img: cv2.bilateralFilter((img * 255).astype(np.uint8), 11, 100, 100).astype(np.float32) / 255.0),
        ("noop",               lambda img: img.copy()),
    ]

    def __init__(self, dataset, max_steps=10, ssim_weight=10.0):
        super().__init__()
        self.dataset = dataset
        self.max_steps = max_steps
        self.ssim_weight = ssim_weight
        h, w, c = dataset[0]["clean"].shape
        self.observation_space = spaces.Box(0.0, 1.0, shape=(c, h, w), dtype=np.float32)
        self.action_space = spaces.Discrete(len(self.FILTER_ACTIONS))

    def _quality(self, img):
        psnr = compute_psnr(self.target, img, data_range=1.0)
        ssim = compute_ssim(self.target, img, data_range=1.0, channel_axis=2)
        return psnr, ssim

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        idx = random.randint(0, len(self.dataset) - 1)
        self.target = self.dataset[idx]["clean"].copy()
        self.current = self.dataset[idx]["noisy"].copy()
        self.step_count = 0
        self.prev_psnr, self.prev_ssim = self._quality(self.current)
        return self.current.transpose(2, 0, 1), {"psnr": self.prev_psnr, "ssim": self.prev_ssim}

    def step(self, action: int):
        _, filter_fn = self.FILTER_ACTIONS[action]
        self.current = np.clip(filter_fn(self.current), 0, 1).astype(np.float32)
        self.step_count += 1

        psnr, ssim = self._quality(self.current)
        reward = (psnr - self.prev_psnr) + self.ssim_weight * (ssim - self.prev_ssim)
        self.prev_psnr, self.prev_ssim = psnr, ssim

        terminated = self.step_count >= self.max_steps
        return self.current.transpose(2, 0, 1), float(reward), terminated, False, {"psnr": psnr, "ssim": ssim}


env = DenoisingEnv(dataset)
obs, info = env.reset()
print(f"Obs shape: {obs.shape}, Actions: {env.action_space.n}")
print(f"Initial PSNR: {info['psnr']:.2f} dB, SSIM: {info['ssim']:.4f}")

## 5 — Filter Comparison Demo

Before RL training, let's see what each filter family does to a noisy image.

In [ ]:
sample = dataset[0]
noisy_demo = sample["noisy"]
clean_demo = sample["clean"]

filter_demos = [
    ("Noisy",           noisy_demo),
    ("Gaussian(3,1.0)", cv2.GaussianBlur(noisy_demo, (3, 3), 1.0)),
    ("Gaussian(5,1.5)", cv2.GaussianBlur(noisy_demo, (5, 5), 1.5)),
    ("Median(3)",       cv2.medianBlur((noisy_demo * 255).astype(np.uint8), 3).astype(np.float32) / 255.0),
    ("Bilateral(7,40)", cv2.bilateralFilter((noisy_demo * 255).astype(np.uint8), 7, 40, 40).astype(np.float32) / 255.0),
    ("Clean (target)",  clean_demo),
]

fig, axes = plt.subplots(1, 6, figsize=(20, 3.5))
for i, (name, img) in enumerate(filter_demos):
    axes[i].imshow(np.clip(img, 0, 1))
    psnr = compute_psnr(clean_demo, np.clip(img, 0, 1).astype(np.float32), data_range=1.0) if name != "Clean (target)" else float('inf')
    ssim = compute_ssim(clean_demo, np.clip(img, 0, 1).astype(np.float32), data_range=1.0, channel_axis=2) if name != "Clean (target)" else 1.0
    axes[i].set_title(f"{name}\nPSNR={psnr:.1f} SSIM={ssim:.3f}", fontsize=8)
    axes[i].axis("off")
plt.suptitle("Single-Filter Comparison on Noisy Image", fontsize=13)
plt.tight_layout()
plt.show()

## Deep Derivation: Noise Estimation and Filter Selection Theory

### Step 1: Noise Level Estimation from a Single Image

**Median Absolute Deviation (MAD) estimator:**

Apply a high-pass filter (e.g., Laplacian) to isolate noise:

$$\hat{\sigma} = \frac{\text{median}(|\nabla^2 I|)}{0.6745}$$

**Why 0.6745?** For Gaussian noise, $\text{MAD} = 0.6745 \sigma$. The median is robust to edges (outliers in the gradient domain).

**Proof of the 0.6745 factor:**

For $X \sim \mathcal{N}(0, \sigma^2)$: $P(|X| \leq m) = 0.5$ gives $m = \sigma \Phi^{-1}(0.75) = 0.6745\sigma$. $\blacksquare$

### Step 2: Oracle Filter Selection

Given a clean image $I^*$ and noisy image $I_n$, the **oracle filter** selects the filter minimizing MSE:

$$a^* = \arg\min_{a \in \mathcal{A}} \|f_a(I_n) - I^*\|^2$$

This is the ceiling performance for any adaptive filter selection method. Our RL agent approximates the oracle by learning $\pi(s) \approx a^*$ from training data.

### Step 3: Sequential Denoising Analysis

After applying filters $f_1, f_2, \ldots, f_T$:

$$I_T = f_T \circ f_{T-1} \circ \cdots \circ f_1(I_{\text{noisy}})$$

**Noise reduction:** Each Gaussian filter with $\sigma_i$ reduces noise by factor $\frac{1}{\sqrt{2\pi \sigma_i^2 k^2}}$ where $k$ is kernel size.

**Detail loss:** Each filter also attenuates signal frequencies above $\frac{1}{2\pi\sigma_i}$.

**Optimal sequence theorem:** For a cascade of $T$ Gaussian filters, the total effective blur is:

$$\sigma_{\text{eff}} = \sqrt{\sigma_1^2 + \sigma_2^2 + \cdots + \sigma_T^2}$$

**Proof (convolution of Gaussians):** $G_{\sigma_1} * G_{\sigma_2} = G_{\sqrt{\sigma_1^2 + \sigma_2^2}}$ since the Fourier transform is $\hat{G}_\sigma(\omega) = e^{-\sigma^2\omega^2/2}$, and $e^{-\sigma_1^2\omega^2/2} \cdot e^{-\sigma_2^2\omega^2/2} = e^{-(\sigma_1^2+\sigma_2^2)\omega^2/2}$. $\blacksquare$

**Implication for RL:** Applying multiple gentle Gaussian filters gives the same blur as one strong filter. The advantage of sequential application is that the agent can **switch to bilateral or median** mid-sequence, adapting to the local content.

### Step 4: Information-Theoretic View of Denoising

The denoising problem is channel coding in reverse:

$$I(I^*; I_n) = h(I^*) - h(I^* | I_n)$$

where $I$ is mutual information, $h$ is differential entropy. For AWGN with variance $\sigma^2$:

$$I(I^*; I_n) = \frac{1}{2}\log\left(1 + \frac{\sigma_{\text{signal}}^2}{\sigma^2}\right) \quad \text{bits/pixel}$$

**Upper bound on PSNR improvement:**

$$\text{PSNR}_{\max} = 10\log_{10}\left(\frac{1}{\sigma^2 \cdot 2^{-2I(I^*; I_n)}}\right)$$

No denoising algorithm can exceed this — it is determined by the mutual information between clean and noisy images. $\blacksquare$

## 6 — DQN Agent & Training

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, in_channels, num_actions):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, 5, stride=2, padding=2), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
        )
        self.head = nn.Sequential(
            nn.Linear(64 * 4 * 4, 256), nn.ReLU(),
            nn.Linear(256, num_actions),
        )

    def forward(self, x):
        x = self.features(x)
        return self.head(x.view(x.size(0), -1))


class ReplayBuffer:
    def __init__(self, capacity):
        self.buf = collections.deque(maxlen=capacity)

    def push(self, *t):
        self.buf.append(t)

    def sample(self, bs):
        batch = random.sample(self.buf, bs)
        s, a, r, ns, d = zip(*batch)
        return (torch.FloatTensor(np.array(s)).to(device),
                torch.LongTensor(a).to(device),
                torch.FloatTensor(r).to(device),
                torch.FloatTensor(np.array(ns)).to(device),
                torch.FloatTensor(d).to(device))

    def __len__(self):
        return len(self.buf)

In [ ]:
NUM_EPISODES = 400
BATCH_SIZE = 64
GAMMA = 0.99
LR = 1e-3
BUFFER_SIZE = 15_000
EPS_START, EPS_END, EPS_DECAY = 1.0, 0.05, 300
TARGET_UPDATE = 10
MAX_STEPS = 10

policy_net = QNetwork(3, env.action_space.n).to(device)
target_net = QNetwork(3, env.action_space.n).to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=LR)
replay_buffer = ReplayBuffer(BUFFER_SIZE)

ep_rewards, ep_psnrs, ep_ssims, losses = [], [], [], []
global_step = 0

for ep in range(NUM_EPISODES):
    obs, info = env.reset()
    ep_reward = 0.0

    for t in range(MAX_STEPS):
        eps = EPS_END + (EPS_START - EPS_END) * math.exp(-global_step / EPS_DECAY)
        if random.random() < eps:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                action = policy_net(torch.FloatTensor(obs).unsqueeze(0).to(device)).argmax(1).item()

        next_obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        replay_buffer.push(obs, action, reward, next_obs, float(done))
        obs = next_obs
        ep_reward += reward
        global_step += 1

        if len(replay_buffer) >= BATCH_SIZE:
            states, actions, rewards, next_states, dones = replay_buffer.sample(BATCH_SIZE)
            q_vals = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                next_q = target_net(next_states).max(1)[0]
                target_q = rewards + GAMMA * next_q * (1 - dones)
            loss = F.smooth_l1_loss(q_vals, target_q)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(policy_net.parameters(), 1.0)
            optimizer.step()
            losses.append(loss.item())

        if done:
            break

    ep_rewards.append(ep_reward)
    ep_psnrs.append(info["psnr"])
    ep_ssims.append(info["ssim"])

    if ep % TARGET_UPDATE == 0:
        target_net.load_state_dict(policy_net.state_dict())

    if ep % 50 == 0:
        print(f"Ep {ep:4d} | Reward: {np.mean(ep_rewards[-50:]):7.2f} | PSNR: {np.mean(ep_psnrs[-50:]):6.2f} | SSIM: {np.mean(ep_ssims[-50:]):.4f} | ε: {eps:.3f}")

print("\nTraining complete!")

## 7 — Training Curves

In [ ]:
def moving_avg(d, w=25):
    return np.convolve(d, np.ones(w)/w, mode="valid")

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

axes[0].plot(ep_rewards, alpha=0.25, color="steelblue")
axes[0].plot(moving_avg(ep_rewards), color="navy", lw=2)
axes[0].set(xlabel="Episode", ylabel="Reward", title="Episode Reward")
axes[0].grid(True, alpha=0.3)

axes[1].plot(ep_psnrs, alpha=0.25, color="coral")
axes[1].plot(moving_avg(ep_psnrs), color="darkred", lw=2)
axes[1].set(xlabel="Episode", ylabel="PSNR (dB)", title="Final PSNR")
axes[1].grid(True, alpha=0.3)

axes[2].plot(ep_ssims, alpha=0.25, color="orchid")
axes[2].plot(moving_avg(ep_ssims), color="purple", lw=2)
axes[2].set(xlabel="Episode", ylabel="SSIM", title="Final SSIM")
axes[2].grid(True, alpha=0.3)

axes[3].plot(losses[:800], alpha=0.25, color="mediumseagreen")
if len(losses) > 25:
    axes[3].plot(moving_avg(losses[:800]), color="darkgreen", lw=2)
axes[3].set(xlabel="Step", ylabel="Loss", title="DQN Loss")
axes[3].grid(True, alpha=0.3)

plt.suptitle("Denoising Agent — Training Dynamics", fontsize=14)
plt.tight_layout()
plt.show()

## 8 — Before / After Evaluation

## Deep Derivation: Bayesian Denoising and Sparse Priors

### Step 1: MAP Denoising Framework

The Maximum A Posteriori estimate:

$$\hat{I}_{\text{MAP}} = \arg\max_I P(I | I_n) = \arg\max_I P(I_n | I) P(I)$$

$$= \arg\min_I \underbrace{\frac{\|I_n - I\|^2}{2\sigma^2}}_{\text{data fidelity}} + \underbrace{(-\log P(I))}_{\text{prior/regularizer}}$$

**Different priors → different denoisers:**

| Prior $P(I)$ | Regularizer $-\log P(I)$ | Denoiser |
|---|---|---|
| Gaussian | $\|I\|^2$ | Wiener filter |
| Laplacian | $\|I\|_1$ | Soft thresholding |
| TV | $\text{TV}(I)$ | ROF model |
| Sparse wavelet | $\|\Psi I\|_1$ | Wavelet shrinkage |

### Step 2: Wavelet Shrinkage — BayesShrink

In the wavelet domain, the MMSE estimate with a generalized Gaussian prior is soft thresholding:

$$\hat{w}_i = \text{sign}(w_i) \max(|w_i| - \tau, 0)$$

**Optimal threshold (Donoho & Johnstone):**

$$\tau^* = \sigma^2 / \sigma_s$$

where $\sigma_s = \sqrt{\max(\sigma_w^2 - \sigma^2, 0)}$ is the estimated signal standard deviation in the wavelet subband.

**Proof soft thresholding is MMSE-optimal for Laplacian prior:**

For $X \sim \text{Laplace}(0, b)$ and $Y = X + N$, $N \sim \mathcal{N}(0, \sigma^2)$:

$$\hat{X}_{\text{MMSE}}(y) = E[X|Y=y] \approx \text{sign}(y)\max(|y| - \sigma^2/b, 0)$$

This is the soft thresholding operator with $\tau = \sigma^2/b$. $\blacksquare$

### Step 3: BM3D — Block Matching and 3D Filtering

The state-of-the-art classical denoiser operates in three steps:

1. **Block matching:** Find similar patches $\{P_i\}_{i=1}^M$ using:
$$d(P_i, P_j) = \frac{\|P_i - P_j\|^2 - 2|P|\sigma^2}{\max(1, \|P_i - P_j\|^2 - 2|P|\sigma^2)}$$

2. **3D transform and shrinkage:** Stack matched patches into a 3D array, apply 3D wavelet transform, threshold coefficients.

3. **Aggregation:** Place filtered patches back and average overlapping regions.

**Why 3D transform works:** The third dimension (across similar patches) exploits **non-local self-similarity** — natural images contain many similar patches, and averaging them reduces noise by $\sqrt{M}$.

### Step 4: RL Agent's Strategy vs. BM3D

The RL agent cannot match BM3D's patch-level sophistication but offers complementary advantages:

$$\text{RL advantage} = \begin{cases} \text{Adaptive filter selection per region} \\ \text{Learned noise-detail trade-off} \\ \text{Sequential refinement capability} \end{cases}$$

The agent learns **when bilateral > Gaussian** (near edges), **when median is best** (salt-and-pepper noise), and **when to stop** (avoid over-smoothing). This meta-level strategy is orthogonal to any single filter's internal mechanism.

In [ ]:
def evaluate_denoising(env, policy_net, n=6):
    results = []
    policy_net.eval()
    for _ in range(n):
        obs, info = env.reset()
        noisy_img = env.current.copy()
        init_psnr, init_ssim = info["psnr"], info["ssim"]
        actions_seq = []
        for t in range(MAX_STEPS):
            with torch.no_grad():
                action = policy_net(torch.FloatTensor(obs).unsqueeze(0).to(device)).argmax(1).item()
            actions_seq.append(action)
            obs, _, done, _, info = env.step(action)
            if done:
                break
        results.append({"noisy": noisy_img, "denoised": env.current.copy(), "clean": env.target.copy(),
                        "psnr_before": init_psnr, "psnr_after": info["psnr"],
                        "ssim_before": init_ssim, "ssim_after": info["ssim"],
                        "actions": actions_seq})
    policy_net.train()
    return results

results = evaluate_denoising(env, policy_net)

fig, axes = plt.subplots(3, 6, figsize=(20, 10))
for i, r in enumerate(results):
    axes[0, i].imshow(r["noisy"])
    axes[0, i].set_title(f"Noisy\nPSNR={r['psnr_before']:.1f}", fontsize=9)
    axes[0, i].axis("off")
    axes[1, i].imshow(np.clip(r["denoised"], 0, 1))
    axes[1, i].set_title(f"Denoised\nPSNR={r['psnr_after']:.1f}", fontsize=9)
    axes[1, i].axis("off")
    axes[2, i].imshow(r["clean"])
    axes[2, i].set_title("Clean", fontsize=9)
    axes[2, i].axis("off")

axes[0, 0].set_ylabel("Noisy Input", fontsize=11)
axes[1, 0].set_ylabel("Agent Output", fontsize=11)
axes[2, 0].set_ylabel("Ground Truth", fontsize=11)
plt.suptitle("Denoising Agent — Before vs After", fontsize=14)
plt.tight_layout()
plt.show()

print(f"{'Img':>4} | {'PSNR Before':>11} | {'PSNR After':>10} | {'SSIM Before':>11} | {'SSIM After':>10}")
print("-" * 62)
for i, r in enumerate(results):
    print(f"{i:4d} | {r['psnr_before']:11.2f} | {r['psnr_after']:10.2f} | {r['ssim_before']:11.4f} | {r['ssim_after']:10.4f}")

## 9 — Learned Filter Selection Strategy

In [ ]:
filter_names = [name for name, _ in DenoisingEnv.FILTER_ACTIONS]

# Collect action statistics
all_actions = []
step_actions = {t: [] for t in range(MAX_STEPS)}
for _ in range(100):
    obs, _ = env.reset()
    policy_net.eval()
    for t in range(MAX_STEPS):
        with torch.no_grad():
            action = policy_net(torch.FloatTensor(obs).unsqueeze(0).to(device)).argmax(1).item()
        all_actions.append(action)
        step_actions[t].append(action)
        obs, _, done, _, _ = env.step(action)
        if done:
            break

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

counts = np.bincount(all_actions, minlength=len(filter_names))
ax1.barh(filter_names, counts, color="teal")
ax1.set_xlabel("Frequency (100 episodes)")
ax1.set_title("Overall Action Distribution")
ax1.grid(True, alpha=0.3, axis="x")

# Heatmap: action distribution per time step
heatmap = np.zeros((len(filter_names), MAX_STEPS))
for t in range(MAX_STEPS):
    if step_actions[t]:
        c = np.bincount(step_actions[t], minlength=len(filter_names))
        heatmap[:, t] = c / max(c.sum(), 1)

im = ax2.imshow(heatmap, aspect="auto", cmap="YlOrRd")
ax2.set_yticks(range(len(filter_names)))
ax2.set_yticklabels(filter_names, fontsize=8)
ax2.set_xlabel("Time Step")
ax2.set_title("Filter Selection Over Episode Steps")
plt.colorbar(im, ax=ax2, label="Probability")

plt.suptitle("Learned Denoising Strategy", fontsize=14)
plt.tight_layout()
plt.show()

## 10 — Key Takeaways

| Insight | Detail |
|---------|--------|
| **Sequential denoising** | Applying multiple gentle filters often outperforms a single aggressive filter |
| **Bilateral preference** | Edge-preserving bilateral filters are typically favoured, especially in later steps |
| **SSIM penalty** | High $\lambda_{\text{SSIM}}$ prevents the agent from over-smoothing |
| **Temporal strategy** | The agent learns to apply stronger filters early and gentler filters later |
| **Extensions** | Non-local means, learned CNN denoisers as actions, noise-level estimation for state |

---

*Next notebook → Module 7.4: Super-Resolution Agent with region-based upscaling strategies.*